In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
import random
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torch.nn.functional as F

############################################
# 1) Алфавит (без изменений)
############################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
num_classes = len(alphabet)  # 45 (0..44, 0=blank)

############################################
# 2) Функция кодирования метки (без изменений)
############################################
def encode_label(text: str, alpha: dict) -> list:
    indices = []
    for ch in text:
        if ch in alpha:
            indices.append(alpha[ch])
    return indices

############################################
# 3) Новая функция препроцессинга аудио
############################################
def preprocess_audio(y, orig_sr, target_sr=16000, top_db=20):
    """
    Выполняет базовую нормализацию и очистку аудиосигнала:
    1) Если частота дискретизации отличается от target_sr, выполняется ресемплирование.
    2) Выравнивается амплитуда аудиосигнала.
    3) Удаляется тишина в начале и конце аудиосигнала.
    4) (Опционально) Можно добавить шумоподавление.
    """
    # Если частота не совпадает, пересчитаем аудиосигнал
    if orig_sr != target_sr:
        y = librosa.resample(y, orig_sr, target_sr)
    
    # Удаляем DC-компоненту
    y = y - np.mean(y)
    
    # Нормализуем амплитуду (приводим значения к диапазону -1...+1)
    y = librosa.util.normalize(y)

    # Обрезаем тишину по краям (можно настроить top_db для регулировки)
    y, _ = librosa.effects.trim(y, top_db=top_db)
    
    # (Опционально) Шумоподавление:
    # Если у вас есть библиотека noisereduce, можно раскомментировать следующие строки:
    # import noisereduce as nr
    # y = nr.reduce_noise(y=y, sr=target_sr)
    
    return y

############################################
# 4) Аудио-аугментация (оставляем или корректируем по необходимости)
############################################
def augment_waveform(y, sr=16000, noise_factor=0.002, stretch_range=(0.95, 1.05), pitch_range=(-1, 1)):
    if random.random() < 0.5:
        noise = np.random.randn(len(y)) * noise_factor
        y = y + noise.astype(np.float32)

    if random.random() < 0.5:
        rate = random.uniform(*stretch_range)
        try:
            y = librosa.effects.time_stretch(y, rate=rate)
        except:
            pass

    if random.random() < 0.3:
        steps = random.uniform(*pitch_range)
        try:
            y = librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)
        except:
            pass

    return y

############################################
# 5) SpecAugment (без изменений)
############################################
def spec_augment(mel, max_freq_mask=15, max_time_mask=30, num_masks=2):
    for _ in range(num_masks):
        f = random.randint(0, max_freq_mask)
        f0 = random.randint(0, max(1, mel.shape[0] - f))
        mel[f0:f0 + f, :] = 0.0

        t = random.randint(0, max_time_mask)
        t0 = random.randint(0, max(1, mel.shape[1] - t))
        mel[:, t0:t0 + t] = 0.0
    return mel

############################################
# 6) Преобразование аудио в Mel-спектрограмму (без изменений, но можно добавить нормализацию там)
############################################
def audio_to_melspectrogram(y, sr=16000, n_mels=128, n_fft=1024, hop_length=512,
                            do_specaug=False):
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    log_S = librosa.power_to_db(S, ref=np.max)
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)

    # Применяем SpecAugment (с вероятностью)
    if do_specaug and random.random() < 0.7:
        log_S_norm = spec_augment(log_S_norm, max_freq_mask=10, max_time_mask=20)

    return log_S_norm  # [n_mels, time]

############################################
# 7) Dataset c препроцессингом аудио
############################################
class MorseAudioCTCDataset(Dataset):
    def __init__(self, df, sr=16000, augment=False, transform=None):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.augment = augment     # если True, применяем augment_waveform
        self.transform = transform # функция (y) -> Mel-спектрограмма

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        audio_path = "morse_dataset/" + row['id']
        # Загружаем аудио с указанием желаемой частоты дискретизации
        y, orig_sr = librosa.load(audio_path, sr=self.sr)
        
        # Применяем препроцессинг:
        y = preprocess_audio(y, orig_sr, target_sr=self.sr)
        
        # Если требуется аугментация, применяем её после препроцессинга
        if self.augment:
            y = augment_waveform(y, sr=self.sr)
        
        # Преобразуем аудио в Mel-спектрограмму (с возможностью SpecAugment)
        if self.transform:
            mel = self.transform(y, sr=self.sr)
        else:
            mel = y

        mel_tensor = torch.tensor(mel, dtype=torch.float)
        # Кодируем метки
        text_label = row['message']
        label_indices = encode_label(text_label, alphabet)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)

        time_dim = mel_tensor.shape[1]
        return mel_tensor, label_tensor, time_dim

############################################
# 8) Функция collate_fn для CTC (без изменений)
############################################
def ctc_collate_fn(batch, pool_time_factor=4):
    mel_list = []
    label_list = []
    tgt_len_list = []
    raw_time_list = []

    for (mel, label, tdim) in batch:
        mel_list.append(mel)
        label_list.append(label)
        tgt_len_list.append(len(label))
        raw_time_list.append(tdim)

    max_time = max(m.shape[1] for m in mel_list)
    padded_mels = []
    for mel in mel_list:
        diff = max_time - mel.shape[1]
        if diff > 0:
            mel = F.pad(mel, (0, diff), value=0.0)
        mel = mel.unsqueeze(0)  # => [1, n_mels, max_time]
        padded_mels.append(mel)

    audio_batch = torch.stack(padded_mels, dim=0)  # [B, 1, n_mels, max_time]
    labels_concat = torch.cat(label_list, dim=0)

    input_lengths = []
    for rt in raw_time_list:
        input_lengths.append(rt // pool_time_factor)
    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(tgt_len_list, dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

############################################
# 9) Модель (без изменений)
############################################
class CNNBiLSTMCTC(nn.Module):
    def __init__(self, num_classes=45, in_channels=1, n_mels=128, lstm_hidden=384, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1),  # [B, 32, n_mels, T]
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # [B, 64, n_mels//2, T//2]

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),  # [B, 128, n_mels//4, T//4]
        )

        self.dropout = nn.Dropout(dropout)

        # После Conv2d и MaxPool: [B, 128, n_mels//4, T//4] -> [B, T//4, 128 * n_mels//4]
        self.rnn_input_dim = (n_mels // 4) * 128
        self.lstm = nn.LSTM(
            input_size=self.rnn_input_dim,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.classifier = nn.Linear(lstm_hidden * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)  # [B, C, F, T]
        b, c, f, t = x.size()
        x = x.permute(0, 3, 1, 2).contiguous().view(b, t, c * f)  # [B, T, C*F]
        x = self.dropout(x)
        x, _ = self.lstm(x)  # [B, T, hidden*2]
        x = self.classifier(x)  # [B, T, num_classes]
        return x.permute(1, 0, 2)  # [T, B, num_classes] — для CTC

############################################
# 10) Цикл обучения (без существенных изменений)
############################################
def train_ctc_loop(model, train_loader, val_loader, num_epochs=15, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        train_loss_sum = 0.0
        train_bar = tqdm(train_loader, desc="Train", leave=False)

        for audio_batch, labels_concat, input_lengths, target_lengths in train_bar:
            audio_batch = audio_batch.to(device)
            labels_concat = labels_concat.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            logits = model(audio_batch)  # [T, B, C]
            log_probs = F.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss_sum / len(train_loader)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_bar = tqdm(val_loader, desc="Val", leave=False)
        with torch.no_grad():
            for audio_batch, labels_concat, input_lengths, target_lengths in val_bar:
                audio_batch = audio_batch.to(device)
                labels_concat = labels_concat.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(audio_batch)
                log_probs = F.log_softmax(logits, dim=2)
                loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
                val_loss_sum += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss_sum / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

############################################
# 11) Основной запуск
############################################
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    # Функция преобразования для train с SpecAugment
    def transform_train(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=True)

    # Функция преобразования для валидации без SpecAugment
    def transform_val(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=False)

    train_dataset = MorseAudioCTCDataset(
        train_df, sr=16000, augment=True, transform=transform_train
    )
    val_dataset = MorseAudioCTCDataset(
        val_df, sr=16000, augment=False, transform=transform_val
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=12,
        shuffle=False,
        num_workers=0,  # увеличиваем число потоков для ускорения загрузки
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=8,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )

    model = CNNBiLSTMCTC(
        num_classes=num_classes,
        lstm_hidden=256,
        lstm_layers=3,
        dropout=0.3
    )

    # Проверяем размеры батча
    audio_batch, labels_concat, input_lengths, target_lengths = next(iter(train_loader))
    print("audio_batch:", audio_batch.shape)
    print("labels_concat:", labels_concat.shape, labels_concat[:10])
    print("input_lengths:", input_lengths)
    print("target_lengths:", target_lengths)

    # Запуск обучения
    train_ctc_loop(
        model,
        train_loader,
        val_loader,
        num_epochs=20,
        lr=1e-3
    )


audio_batch: torch.Size([12, 1, 128, 264])
labels_concat: torch.Size([102]) tensor([29, 16, 37,  5, 11, 33, 35, 36, 13, 38])
input_lengths: tensor([62, 62, 61, 62, 62, 66, 62, 62, 62, 62, 63, 62])
target_lengths: tensor([ 6,  7, 10,  9, 10, 11,  8, 10,  7,  8,  9,  7])

Epoch 1/20


KeyboardInterrupt: 

In [23]:
torch.save(model.state_dict(), "w_aug.pth")

In [26]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import pandas as pd
import numpy as np
import scipy.signal

#########################################
# Алфавит
#########################################
alphabet = {
    '<pad>': 0, ' ': 1, '#': 2,
    '0': 3, '1': 4, '2': 5, '3': 6, '4': 7, '5': 8, '6': 9, '7': 10, '8': 11, '9': 12,
    'А': 13, 'Б': 14, 'В': 15, 'Г': 16, 'Д': 17, 'Е': 18, 'Ж': 19, 'З': 20,
    'И': 21, 'Й': 22, 'К': 23, 'Л': 24, 'М': 25, 'Н': 26, 'О': 27, 'П': 28,
    'Р': 29, 'С': 30, 'Т': 31, 'У': 32, 'Ф': 33, 'Х': 34, 'Ц': 35, 'Ч': 36,
    'Ш': 37, 'Щ': 38, 'Ъ': 39, 'Ы': 40, 'Ь': 41, 'Э': 42, 'Ю': 43, 'Я': 44
}
idx2char = {v: k for k, v in alphabet.items()}

#########################################
# Модель (такая же как при обучении)
#########################################
class EnhancedCTCModel(nn.Module):
    """
    Два Conv+Pool -> BiLSTM -> Linear.
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=64,
                 hidden_size=256, lstm_layers=3, dropout=0.3):
        super(EnhancedCTCModel, self).__init__()

        # Блок 1: Conv -> BN -> ReLU -> Pool
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        # Блок 2: Conv -> BN -> ReLU -> Pool
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )
        # LSTM: вход = 32 * (n_mels//4)
        self.lstm = nn.LSTM(
            input_size=32 * (n_mels // 4),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=(dropout if lstm_layers > 1 else 0.0),
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels=64, time] => [time//4, B, num_classes]
        """
        x = self.conv1(x)
        x = self.conv2(x)
        b, c, f, t = x.shape
        x = x.view(b, c * f, t)
        x = x.permute(2, 0, 1)  # [T, B, feature]
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits  # [T, B, num_classes]

#########################################
# Преобразование аудио -> Mel (с нормализацией и фильтром)
#########################################
def normalize_audio(y):
    return y / (np.max(np.abs(y)) + 1e-6)

def bandpass_filter(y, sr=16000, low=300, high=3000):
    sos = scipy.signal.butter(4, [low, high], btype='bandpass', fs=sr, output='sos')
    return scipy.signal.sosfilt(sos, y)

def preprocess_test_audio(y, sr=16000):
    y = normalize_audio(y)
    y = bandpass_filter(y, sr=sr)
    return y

def audio_to_melspectrogram(y, sr=16000, n_mels=128, n_fft=1024, hop_length=512):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft,
                                       hop_length=hop_length, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm

#########################################
# Greedy decode
#########################################
def ctc_greedy_decode(logits, blank=0):
    argmax = torch.argmax(logits, dim=2)  # [T, 1]
    argmax = argmax.squeeze(1).cpu().numpy().tolist()
    decoded = []
    prev = None
    for idx in argmax:
        if idx != blank and idx != prev:
            decoded.append(idx)
        prev = idx
    return decoded

#########################################
# Инференс + submit
#########################################
def infer_greedy(
    model_path="w_out_aug.pth",
    test_csv="test.csv",
    audio_dir="morse_dataset",
    output_csv="submission_7.csv"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = EnhancedCTCModel(
        num_classes=45,
        in_channels=1,
        n_mels=128,
        hidden_size=256,
        lstm_layers=3,
        dropout=0.3
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    df_test = pd.read_csv(test_csv)
    results = []

    for idx, row in df_test.iterrows():
        file_id = row['id']
        audio_path = os.path.join(audio_dir, file_id)

        y, _ = librosa.load(audio_path, sr=16000)
        y = preprocess_test_audio(y, sr=16000)
        mel = audio_to_melspectrogram(y, sr=16000, n_mels=128)  # [n_mels=64, time]
        mel_tensor = torch.tensor(mel, dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(mel_tensor)  # [T, 1, 45]
            log_probs = F.log_softmax(logits, dim=2)

        pred_indices = ctc_greedy_decode(log_probs, blank=0)
        pred_text = ''.join(idx2char[i] for i in pred_indices if i in idx2char)

        results.append({
            "id": file_id,
            "message": pred_text
        })

    df_sub = pd.DataFrame(results)
    df_sub.to_csv(output_csv, index=False)
    print(f"✅ Saved submission: {output_csv}")

#########################################
# Запуск
#########################################
if __name__ == "__main__":
    infer_greedy(
        model_path="w_aug.pth",
        test_csv="test.csv",
        audio_dir="morse_dataset",
        output_csv="submission_8.csv"
    )


✅ Saved submission: submission_8.csv


ModuleNotFoundError: No module named 'ctcdecode'